In [0]:
source = 'allscripts_tw'

In [0]:

dbo_person_address_df = spark.sql(f'''SELECT 
LOWER(TRIM(dbo_person_address.addressline1)) AS address_1,
LOWER(TRIM(dbo_person_address.addressline2)) AS address_2,
LOWER(TRIM(dbo_person_address.city)) AS city,
LOWER(TRIM(dbo_person_address.state)) AS state,
LOWER(TRIM(dbo_person_address.zipcode)) zip,
LOWER(TRIM(dbo_person_address.county)) AS county,
CONCAT_WS(CHR(31), '{source}','dbo_person_address', 'id', CAST(dbo_person_address.id AS BIGINT)) AS location_source_value,
COALESCE(domain_source_to_concept.omop_concept_id, 0) AS country_concept_id,
LOWER(TRIM(dbo_person_address.country)) AS country_source_value,
NULL AS latitude,
NULL AS longitude,
'{source}' AS source_system
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_person_address
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept
ON LOWER(TRIM(dbo_person_address.country)) = LOWER(TRIM(domain_source_to_concept.source_value))
AND LOWER(domain_source_to_concept.source_system) = 'allscripts_tw' 
AND domain_source_to_concept.source_table = 'dbo_person_address' 
AND LOWER(domain_source_to_concept.source_field) = 'country' 
AND domain_source_to_concept.domain_id = 'Geography'
WHERE 1=1
AND dbo_person_address.addressline1 IS NOT NULL
''')
silver_df = dbo_person_address_df
silver_df.createOrReplaceTempView("silver")


In [0]:
%sql
MERGE INTO _exponent.omop_silver.location AS target
USING silver AS source
ON target.location_source_value = source.location_source_value

WHEN MATCHED AND NOT (
     target.address_1              <=> source.address_1
 AND target.address_2              <=> source.address_2
 AND target.city                   <=> source.city
 AND target.state                  <=> source.state
 AND target.zip                    <=> source.zip
 AND target.county                 <=> source.county
 AND target.country_concept_id     <=> source.country_concept_id
 AND target.country_source_value   <=> source.country_source_value
 AND target.latitude               <=> source.latitude
 AND target.longitude              <=> source.longitude
 AND target.source_system          <=> source.source_system
) THEN UPDATE SET
  target.address_1              = source.address_1,
  target.address_2              = source.address_2,
  target.city                   = source.city,
  target.state                  = source.state,
  target.zip                    = source.zip,
  target.county                 = source.county,
  target.country_concept_id     = source.country_concept_id,
  target.country_source_value   = source.country_source_value,
  target.latitude               = source.latitude,
  target.longitude              = source.longitude,
  target.source_system          = source.source_system,
  target.last_mod_tsp           = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  address_1,
  address_2,
  city,
  state,
  zip,
  county,
  location_source_value,
  country_concept_id,
  country_source_value,
  latitude,
  longitude,
  source_system,
  last_mod_tsp
) VALUES (
  source.address_1,
  source.address_2,
  source.city,
  source.state,
  source.zip,
  source.county,
  source.location_source_value,
  source.country_concept_id,
  source.country_source_value,
  source.latitude,
  source.longitude,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_location (
    source_system,
    location_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    silver_location.source_system,
    silver_location.location_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(silver_location.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        location_source_value,
        last_mod_tsp
    FROM _exponent.omop_silver.location
    WHERE location_source_value IS NOT NULL
) AS silver_location
LEFT ANTI JOIN _exponent.omop_mapping.source_to_location AS existing_location
  ON silver_location.location_source_value = existing_location.location_source_value;

In [0]:
gold_df = spark.sql("""
SELECT
  source_to_location.location_id,
  location.address_1,
  location.address_2,
  location.city,
  location.state,
  location.zip,
  location.county,
  location.country_concept_id,
  location.latitude,
  location.longitude,
  location.location_source_value,
  location.last_mod_tsp
FROM _exponent.omop_silver.location AS location
JOIN _exponent.omop_mapping.source_to_location AS source_to_location
  ON location.location_source_value = source_to_location.location_source_value
 AND source_to_location.active_flag = TRUE
""")

gold_df.createOrReplaceTempView("gold")

In [0]:
%sql
    
MERGE INTO _exponent.omop.location AS target
USING gold AS source
ON target.location_id = source.location_id

WHEN MATCHED AND NOT (
     target.address_1              <=> source.address_1
 AND target.address_2              <=> source.address_2
 AND target.city                   <=> source.city
 AND target.state                  <=> source.state
 AND target.zip                    <=> source.zip
 AND target.county                 <=> source.county
 AND target.country_concept_id     <=> source.country_concept_id
 AND target.latitude               <=> source.latitude
 AND target.longitude              <=> source.longitude
 AND target.location_source_value  <=> source.location_source_value
) THEN UPDATE SET
  target.address_1               = source.address_1,
  target.address_2               = source.address_2,
  target.city                    = source.city,
  target.state                   = source.state,
  target.zip                     = source.zip,
  target.county                  = source.county,
  target.country_concept_id      = source.country_concept_id,
  target.latitude                = source.latitude,
  target.longitude               = source.longitude,
  target.location_source_value   = source.location_source_value

WHEN NOT MATCHED THEN INSERT (
  location_id,
  address_1,
  address_2,
  city,
  state,
  zip,
  county,
  location_source_value,
  country_concept_id,
  latitude,
  longitude
) VALUES (
  source.location_id,
  source.address_1,
  source.address_2,
  source.city,
  source.state,
  source.zip,
  source.county,
  source.location_source_value,
  source.country_concept_id,
  source.latitude,
  source.longitude
);